# Download the MEVID dataset
Run this notebook with a local Windows Python kernel. It downloads the annotation ZIP and train/test image archives into `D:\Study\ImageProcess\MultiCameraTracking\data`. The archives are large, so check that the drive has enough free space before starting. Completed extractions are skipped on reruns. A Colab runtime cannot access your local D: drive.

In [ ]:
from pathlib import Path
import shutil
import tarfile
import time
import urllib.request
import zipfile

DATA_DIR = Path(r'D:\Study\ImageProcess\MultiCameraTracking\data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset directory: {DATA_DIR}')
print(f'Free disk space: {shutil.disk_usage(DATA_DIR).free / 1024**3:.1f} GiB')

In [3]:
BASE_URL = 'https://mevadata-public-01.s3.amazonaws.com/mevid-annotations'

def download(url, destination):
    partial = destination.with_name(destination.name + '.part')
    offset = partial.stat().st_size if partial.exists() else 0
    request = urllib.request.Request(url, headers={'Range': f'bytes={offset}-'} if offset else {})
    with urllib.request.urlopen(request) as response:
        mode = 'ab' if offset and response.status == 206 else 'wb'
        downloaded = offset if mode == 'ab' else 0
        length = int(response.headers.get('Content-Length', 0))
        total = downloaded + length if length else 0
        started = time.monotonic()
        start_bytes = downloaded
        next_percent = (downloaded * 10 // total + 1) * 10 if total else None
        if downloaded:
            print(f'Resuming from {downloaded / 1024**3:.2f} GiB', flush=True)
        with partial.open(mode) as output:
            while chunk := response.read(1024 * 1024):
                output.write(chunk)
                downloaded += len(chunk)
                now = time.monotonic()
                if total and downloaded * 100 >= next_percent * total:
                    speed = (downloaded - start_bytes) / max(now - started, 0.001)
                    eta = f', ETA {(total - downloaded) / speed / 60:.1f} min' if speed else ''
                    print(f'{destination.name}: {next_percent}% ({downloaded / 1024**3:.2f} GiB), {speed / 1024**2:.1f} MiB/s{eta}', flush=True)
                    next_percent += 10
        if total and downloaded != total:
            raise IOError(f'Incomplete download: {downloaded} of {total} bytes; rerun to resume')
    partial.replace(destination)
ARCHIVES = [
    ('Annotations', 'mevid-v1-annotation-data.zip'),
    ('Train images', 'mevid-v1-bbox-train.tgz'),
    ('Test images', 'mevid-v1-bbox-test.tgz'),
]

for label, filename in ARCHIVES:
    archive = DATA_DIR / filename
    marker = DATA_DIR / f'.{filename}.extracted'
    if marker.exists():
        print(f'{label}: already extracted; skipping')
        continue
    print(f'{label}: downloading {filename}...', flush=True)
    if not archive.exists():
        download(f'{BASE_URL}/{filename}', archive)
    print(f'{label}: extracting...', flush=True)
    if filename.endswith('.zip'):
        with zipfile.ZipFile(archive) as source:
            source.extractall(DATA_DIR)
    else:
        with tarfile.open(archive, 'r:gz') as source:
            source.extractall(DATA_DIR, filter='data')
    marker.touch()
    archive.unlink()
    print(f'{label}: done', flush=True)

print(f'MEVID download complete: {DATA_DIR}')

Annotations: already extracted; skipping
Train images: downloading mevid-v1-bbox-train.tgz...
Resuming from 1.90 GiB
mevid-v1-bbox-train.tgz: 10% (3.04 GiB), 6.2 MiB/s, ETA 75.9 min
mevid-v1-bbox-train.tgz: 20% (6.09 GiB), 6.1 MiB/s, ETA 68.0 min
mevid-v1-bbox-train.tgz: 30% (9.13 GiB), 6.1 MiB/s, ETA 59.5 min
mevid-v1-bbox-train.tgz: 40% (12.18 GiB), 6.1 MiB/s, ETA 51.0 min
mevid-v1-bbox-train.tgz: 50% (15.22 GiB), 6.1 MiB/s, ETA 42.3 min
mevid-v1-bbox-train.tgz: 60% (18.26 GiB), 6.1 MiB/s, ETA 34.1 min
mevid-v1-bbox-train.tgz: 70% (21.31 GiB), 6.2 MiB/s, ETA 25.2 min
mevid-v1-bbox-train.tgz: 80% (24.35 GiB), 6.2 MiB/s, ETA 16.8 min
mevid-v1-bbox-train.tgz: 90% (27.40 GiB), 6.2 MiB/s, ETA 8.3 min
mevid-v1-bbox-train.tgz: 100% (30.44 GiB), 6.2 MiB/s, ETA 0.0 min
Train images: extracting...
Train images: done
Test images: downloading mevid-v1-bbox-test.tgz...
mevid-v1-bbox-test.tgz: 10% (1.30 GiB), 6.9 MiB/s, ETA 29.1 min
mevid-v1-bbox-test.tgz: 20% (2.60 GiB), 6.9 MiB/s, ETA 25.6 min
m